# Transformer Baseline for Synthetic EHR Generation on MIMIC-III

This notebook demonstrates how to train a Transformer-based generative model on MIMIC-III data and generate synthetic patient records.

## Overview
- **Model**: TransformerEHRGenerator (decoder-only transformer, GPT-style)
- **Dataset**: MIMIC-III diagnosis codes
- **Output**: CSV file with columns: `SUBJECT_ID`, `VISIT_NUM`, `ICD9_CODE`

## Setup
Designed for Google Colab with GPU support. Estimated runtime:
- Demo (5 epochs): ~20-30 minutes
- Full training (50 epochs): ~4-6 hours

## 1. Environment Setup

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = "cuda"
else:
    print("WARNING: Running on CPU. Training will be very slow.")
    device = "cpu"

In [ ]:
# Install PyHealth (if not already installed)
# Uncomment the following line if you need to install PyHealth
# !pip install pyhealth

In [ ]:
# Mount Google Drive (optional - for persistent storage)
from google.colab import drive
drive.mount('/content/drive')

# Set paths for persistent storage
DRIVE_ROOT = "/content/drive/MyDrive/PyHealth_Synthetic_EHR"
!mkdir -p "{DRIVE_ROOT}"
!mkdir -p "{DRIVE_ROOT}/data"
!mkdir -p "{DRIVE_ROOT}/models"
!mkdir -p "{DRIVE_ROOT}/output"

## 2. Configuration

In [ ]:
# Configuration parameters
class Config:
    # Paths
    MIMIC_ROOT = f"{DRIVE_ROOT}/data/mimic3"  # Update this to your MIMIC-III path
    OUTPUT_DIR = f"{DRIVE_ROOT}/output"
    MODEL_SAVE_PATH = f"{DRIVE_ROOT}/models/transformer_ehr_best.pth"
    
    # Dataset parameters
    MIN_VISITS = 2  # Minimum visits per patient
    MAX_VISITS = None  # Maximum visits to include (None = all)
    
    # Model architecture
    EMBEDDING_DIM = 256
    NUM_HEADS = 8
    NUM_LAYERS = 6
    DIM_FEEDFORWARD = 1024
    DROPOUT = 0.1
    MAX_SEQ_LENGTH = 512
    
    # Training parameters
    EPOCHS = 5  # Set to 50-80 for production
    BATCH_SIZE = 64  # Reduce to 32 if OOM errors occur
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    
    # Data split
    TRAIN_RATIO = 0.8
    VAL_RATIO = 0.1
    TEST_RATIO = 0.1
    
    # Generation parameters
    NUM_SYNTHETIC_SAMPLES = 1000  # Set to 10000 for production
    MAX_GEN_VISITS = 10
    MAX_CODES_PER_VISIT = 20
    TEMPERATURE = 1.0
    TOP_K = 50
    TOP_P = 0.95

config = Config()
print("Configuration loaded successfully!")
print(f"Training for {config.EPOCHS} epochs")
print(f"Will generate {config.NUM_SYNTHETIC_SAMPLES} synthetic patients")

## 3. Data Upload

Upload your MIMIC-III data files to the specified directory. You need:
- `ADMISSIONS.csv`
- `DIAGNOSES_ICD.csv`

These files should be placed in the directory specified by `config.MIMIC_ROOT`.

In [ ]:
# Check if MIMIC-III files exist
import os

required_files = ['ADMISSIONS.csv', 'DIAGNOSES_ICD.csv']
files_exist = all(os.path.exists(os.path.join(config.MIMIC_ROOT, f)) for f in required_files)

if files_exist:
    print("✓ All required MIMIC-III files found!")
else:
    print("✗ Missing MIMIC-III files. Please upload:")
    for f in required_files:
        path = os.path.join(config.MIMIC_ROOT, f)
        status = "✓" if os.path.exists(path) else "✗"
        print(f"  {status} {f}")
    print(f"\nUpload files to: {config.MIMIC_ROOT}")

## 4. Load and Preprocess Data

In [ ]:
# Import PyHealth modules
from pyhealth.datasets import MIMIC3Dataset
from pyhealth.tasks import SyntheticEHRGenerationMIMIC3
from pyhealth.datasets import split_by_patient, get_dataloader

print("Loading MIMIC-III dataset...")
# Load base dataset
base_dataset = MIMIC3Dataset(
    root=config.MIMIC_ROOT,
    tables=["DIAGNOSES_ICD"],
    code_mapping=None,  # Use raw ICD9 codes
)

print(f"Loaded {len(base_dataset.patients)} patients")

# Apply synthetic EHR generation task
print(f"\nApplying task with min_visits={config.MIN_VISITS}...")
task = SyntheticEHRGenerationMIMIC3(
    min_visits=config.MIN_VISITS,
    max_visits=config.MAX_VISITS
)
sample_dataset = base_dataset.set_task(task)

print(f"Created {len(sample_dataset)} samples")

# Split by patient to prevent data leakage
print(f"\nSplitting data: {config.TRAIN_RATIO}/{config.VAL_RATIO}/{config.TEST_RATIO}")
train_dataset, val_dataset, test_dataset = split_by_patient(
    sample_dataset, 
    [config.TRAIN_RATIO, config.VAL_RATIO, config.TEST_RATIO]
)

print(f"Train: {len(train_dataset)} samples")
print(f"Val: {len(val_dataset)} samples")
print(f"Test: {len(test_dataset)} samples")

In [ ]:
# Create data loaders
print("Creating data loaders...")
train_loader = get_dataloader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True
)
val_loader = get_dataloader(
    val_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False
)
test_loader = get_dataloader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Inspect a sample batch
sample_batch = next(iter(train_loader))
print("Sample batch structure:")
for key, value in sample_batch.items():
    if isinstance(value, torch.Tensor):
        print(f"  {key}: shape {value.shape}, dtype {value.dtype}")
    else:
        print(f"  {key}: {type(value)}")

## 5. Initialize Model

In [ ]:
from pyhealth.models import TransformerEHRGenerator

print("Initializing TransformerEHRGenerator...")
model = TransformerEHRGenerator(
    dataset=sample_dataset,
    embedding_dim=config.EMBEDDING_DIM,
    num_heads=config.NUM_HEADS,
    num_layers=config.NUM_LAYERS,
    dim_feedforward=config.DIM_FEEDFORWARD,
    dropout=config.DROPOUT,
    max_seq_length=config.MAX_SEQ_LENGTH
)

# Move model to device
model = model.to(device)

# Print model info
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel initialized successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Vocabulary size: {model.vocab_size}")
print(f"Device: {device}")

## 6. Training

In [ ]:
from pyhealth.trainer import Trainer

print(f"Starting training for {config.EPOCHS} epochs...\n")

# Initialize trainer
trainer = Trainer(
    model=model,
    device=device,
    output_path=config.OUTPUT_DIR,
    exp_name="transformer_ehr_generator"
)

# Train model
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    epochs=config.EPOCHS,
    monitor="loss",
    monitor_criterion="min",
    load_best_model_at_last=True
)

print("\n" + "="*50)
print("Training completed!")
print("="*50)

In [ ]:
# Save the best model
torch.save(model.state_dict(), config.MODEL_SAVE_PATH)
print(f"Model saved to: {config.MODEL_SAVE_PATH}")

## 7. Evaluation on Test Set

In [ ]:
# Evaluate on test set
print("Evaluating on test set...")
test_results = trainer.evaluate(test_loader)

print("\nTest Results:")
for metric, value in test_results.items():
    print(f"  {metric}: {value:.4f}")

## 8. Generate Synthetic Patients

In [ ]:
# Generate synthetic patient histories
print(f"Generating {config.NUM_SYNTHETIC_SAMPLES} synthetic patients...\n")

model.eval()
with torch.no_grad():
    synthetic_nested_codes = model.generate(
        num_samples=config.NUM_SYNTHETIC_SAMPLES,
        max_visits=config.MAX_GEN_VISITS,
        max_codes_per_visit=config.MAX_CODES_PER_VISIT,
        temperature=config.TEMPERATURE,
        top_k=config.TOP_K,
        top_p=config.TOP_P
    )

print(f"Generated {len(synthetic_nested_codes)} synthetic patients")
print(f"\nExample synthetic patient (first 3 visits):")
if len(synthetic_nested_codes) > 0 and len(synthetic_nested_codes[0]) > 0:
    for i, visit in enumerate(synthetic_nested_codes[0][:3]):
        print(f"  Visit {i+1}: {visit[:10]}{'...' if len(visit) > 10 else ''}")

## 9. Convert to DataFrame Format

In [ ]:
import pandas as pd

# Get the processor to convert token IDs back to codes
input_processor = sample_dataset.input_processors["visit_codes"]
index_to_code = {v: k for k, v in input_processor.code_vocab.items()}

print("Converting synthetic data to CSV format...")

# Convert nested codes to tabular format
rows = []
for patient_idx, patient_visits in enumerate(synthetic_nested_codes):
    synthetic_subject_id = f"SYNTHETIC_{patient_idx:06d}"
    
    for visit_num, visit_codes in enumerate(patient_visits, start=1):
        for code_idx in visit_codes:
            # Convert token ID to actual code
            code = index_to_code.get(code_idx, str(code_idx))
            
            # Skip special tokens
            if code in ['<pad>', '<bos>', '<eos>', 'VISIT_DELIM']:
                continue
            
            rows.append({
                'SUBJECT_ID': synthetic_subject_id,
                'VISIT_NUM': visit_num,
                'ICD9_CODE': code
            })

# Create DataFrame
synthetic_df = pd.DataFrame(rows)

print(f"\nCreated DataFrame with {len(synthetic_df)} rows")
print(f"Number of unique patients: {synthetic_df['SUBJECT_ID'].nunique()}")
print(f"Number of unique codes: {synthetic_df['ICD9_CODE'].nunique()}")

# Display sample
print("\nSample of synthetic data:")
print(synthetic_df.head(20))

## 10. Validation and Quality Checks

In [ ]:
# Quality checks
print("Data Quality Checks:")
print("="*50)

# Check for null values
null_counts = synthetic_df.isnull().sum()
print(f"\n1. Null values:")
for col, count in null_counts.items():
    status = "✓" if count == 0 else "✗"
    print(f"   {status} {col}: {count}")

# Check visit numbering
print(f"\n2. Visit numbering:")
visit_check = synthetic_df.groupby('SUBJECT_ID')['VISIT_NUM'].apply(list)
sequential = all(visits == list(range(1, len(visits)+1)) for visits in visit_check)
print(f"   {'✓' if sequential else '✗'} All visits numbered sequentially")

# Statistics
print(f"\n3. Statistics:")
visits_per_patient = synthetic_df.groupby('SUBJECT_ID')['VISIT_NUM'].max()
codes_per_visit = synthetic_df.groupby(['SUBJECT_ID', 'VISIT_NUM']).size()

print(f"   Visits per patient:")
print(f"     Mean: {visits_per_patient.mean():.2f}")
print(f"     Median: {visits_per_patient.median():.2f}")
print(f"     Min: {visits_per_patient.min()}")
print(f"     Max: {visits_per_patient.max()}")

print(f"   Codes per visit:")
print(f"     Mean: {codes_per_visit.mean():.2f}")
print(f"     Median: {codes_per_visit.median():.2f}")
print(f"     Min: {codes_per_visit.min()}")
print(f"     Max: {codes_per_visit.max()}")

# Code format check
print(f"\n4. Code format:")
sample_codes = synthetic_df['ICD9_CODE'].head(10).tolist()
print(f"   Sample codes: {sample_codes}")

print("\n" + "="*50)
print("Quality checks completed!")
print("="*50)

## 11. Save CSV File

In [ ]:
# Save to CSV
output_csv_path = f"{config.OUTPUT_DIR}/synthetic_ehr_transformer.csv"
synthetic_df.to_csv(output_csv_path, index=False)

print(f"Synthetic data saved to: {output_csv_path}")
print(f"\nFile info:")
print(f"  Rows: {len(synthetic_df):,}")
print(f"  Columns: {list(synthetic_df.columns)}")
print(f"  File size: {os.path.getsize(output_csv_path) / 1024:.2f} KB")

## 12. Download Results

In [ ]:
# Download the CSV file (for Google Colab)
from google.colab import files

print("Preparing download...")
files.download(output_csv_path)
print("Download started!")

## 13. Summary

In [ ]:
# Print final summary
print("\n" + "="*60)
print("SYNTHETIC EHR GENERATION SUMMARY")
print("="*60)

print(f"\nModel: TransformerEHRGenerator")
print(f"  - Embedding dim: {config.EMBEDDING_DIM}")
print(f"  - Layers: {config.NUM_LAYERS}")
print(f"  - Attention heads: {config.NUM_HEADS}")
print(f"  - Parameters: {total_params:,}")

print(f"\nTraining:")
print(f"  - Epochs: {config.EPOCHS}")
print(f"  - Batch size: {config.BATCH_SIZE}")
print(f"  - Training samples: {len(train_dataset)}")
print(f"  - Validation samples: {len(val_dataset)}")

print(f"\nGeneration:")
print(f"  - Synthetic patients: {synthetic_df['SUBJECT_ID'].nunique()}")
print(f"  - Total diagnosis records: {len(synthetic_df)}")
print(f"  - Unique ICD-9 codes: {synthetic_df['ICD9_CODE'].nunique()}")
print(f"  - Avg visits per patient: {visits_per_patient.mean():.2f}")
print(f"  - Avg codes per visit: {codes_per_visit.mean():.2f}")

print(f"\nOutput:")
print(f"  - CSV file: {output_csv_path}")
print(f"  - Model checkpoint: {config.MODEL_SAVE_PATH}")

print("\n" + "="*60)
print("Pipeline completed successfully!")
print("="*60)